# ADM and BSSN Conversions

Convert a simple ADM data set to BSSN variables and check an inverse conversion on a toy metric.

Navigation: [Index](../index.ipynb) | Previous: [ADM 3+1 Background](adm_3plus1_background.ipynb) | Next: [Index](../index.ipynb)

## Why This Matters
BSSN rewrites ADM variables into a form commonly used for stable numerical evolution, so conversions must preserve the represented geometry.

## What You Will See
You will inspect printed expressions, generated artifacts, or diagnostic tables and then connect them to the next notebook.

Prerequisite bridge: this notebook follows [ADM 3+1 Background](adm_3plus1_background.ipynb). Terms are defined here before they are used.

## Convert ADM Variables and Check a Toy Inverse
The conformal factor records the scale removed from the spatial metric. The residual check verifies that a conformally flat toy metric survives an ADM-to-BSSN-to-ADM round trip.

In [1]:
import sympy as sp

import nrpy.indexedexp as ixp
import nrpy.params as par
from nrpy.equations.general_relativity.ADM_to_BSSN import ADM_to_BSSN
from nrpy.equations.general_relativity.BSSN_to_ADM import BSSN_to_ADM
from nrpy.equations.general_relativity.InitialData_Cartesian import InitialData_Cartesian

par.set_parval_from_str("EvolvedConformalFactor_cf", "W")
print("conformal factor choice:", par.parval_from_str("EvolvedConformalFactor_cf"))

brill_lindquist = InitialData_Cartesian("BrillLindquist")
adm_to_bssn = ADM_to_BSSN(brill_lindquist.gammaDD, brill_lindquist.KDD, brill_lindquist.betaU, brill_lindquist.BU, "Cartesian")
print("cf:", adm_to_bssn.cf)
print("hDD[0][0]:", adm_to_bssn.hDD[0][0])
print("hDD[0][1]:", adm_to_bssn.hDD[0][1])
print("aDD[0][0]:", adm_to_bssn.aDD[0][0])
print("trK:", adm_to_bssn.trK)
print("vetU:", adm_to_bssn.vetU)
print("betU:", adm_to_bssn.betU)

psi = sp.symbols("psi", positive=True)
toy_gammaDD = ixp.zerorank2()
toy_KDD = ixp.zerorank2()
toy_betaU = ixp.zerorank1()
toy_BU = ixp.zerorank1()
for i in range(3):
    toy_gammaDD[i][i] = psi**4

toy_adm_to_bssn = ADM_to_BSSN(toy_gammaDD, toy_KDD, toy_betaU, toy_BU, "Cartesian")
toy_bssn_to_adm = BSSN_to_ADM("Cartesian")
substitutions = {}
generic_expressions = []
for i in range(3):
    generic_expressions.extend(toy_bssn_to_adm.gammaDD[i])
    generic_expressions.extend(toy_bssn_to_adm.KDD[i])
generic_expressions.extend(toy_bssn_to_adm.betaU)
for expression in generic_expressions:
    for symbol in expression.free_symbols:
        name = symbol.name
        if name.startswith("hDD") or name.startswith("aDD") or name == "trK":
            substitutions[symbol] = 0
        elif name == "cf":
            substitutions[symbol] = toy_adm_to_bssn.cf
        elif name.startswith("vetU") or name.startswith("betU"):
            substitutions[symbol] = 0

metric_residuals = [sp.factor(toy_bssn_to_adm.gammaDD[i][i].xreplace(substitutions) - toy_gammaDD[i][i]) for i in range(3)]
curvature_residuals = [sp.factor(toy_bssn_to_adm.KDD[i][i].xreplace(substitutions) - toy_KDD[i][i]) for i in range(3)]
shift_residuals = [sp.factor(toy_bssn_to_adm.betaU[i].xreplace(substitutions) - toy_betaU[i]) for i in range(3)]
print("toy cf:", toy_adm_to_bssn.cf)
print("toy hDD[0][0]:", toy_adm_to_bssn.hDD[0][0])
print("toy aDD[0][0]:", toy_adm_to_bssn.aDD[0][0])
print("toy trK:", toy_adm_to_bssn.trK)
print("metric diagonal residuals:", metric_residuals)
print("KDD diagonal residuals:", curvature_residuals)
print("shift residuals:", shift_residuals)
assert metric_residuals == [0, 0, 0]
assert curvature_residuals == [0, 0, 0]
assert shift_residuals == [0, 0, 0]

conformal factor choice: W
Setting up reference_metric[Cartesian]...
cf: ((BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**12)**(-1/6)
hDD[0][0]: (BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**4*((BH1_mass/(2*sqrt((-BH1_posn_x + x)**2 + (-BH1_posn_y + y)**2 + (-BH1_posn_z + z)**2)) + BH2_mass/(2*sqrt((-BH2_posn_x + x)**2 + (-BH2_posn_y + y)**2 + (-BH2_posn_z + z)**2)) + 1)**(-12))**(1/3) - 1
hDD[0][1]: 0
aDD[0][0]: 0
trK: 0
vetU: [0, 0, 0]
betU: [0, 0, 0]
Setting up BSSN_Quantities[Cartesian]...


toy cf: psi**(-2)
toy hDD[0][0]: 0
toy aDD[0][0]: 0
toy trK: 0
metric diagonal residuals: [0, 0, 0]
KDD diagonal residuals: [0, 0, 0]
shift residuals: [0, 0, 0]


## Interpreting the Output
The determinant and reconstruction residuals confirm that the ADM-to-BSSN conversion preserves the represented geometry in this example. The same consistency checks matter before generating evolution code.

## Where This Leads
- [ADM 3+1 Background](adm_3plus1_background.ipynb)
- [Indexed Expressions](../1-intro/indexedexp.ipynb)
- [Basis Transforms](../4-curvilinear/basis_transforms.ipynb)